In [1]:
## 基本工具定义
from langchain.tools import tool

# 装饰器 -- 下面单引号包裹的内容(原来是解释函数的)变为工具的描述，帮助模型理解
# 工具名称(即函数名称)选择用蛇形命名法(snake_case)，即字母、数字字符、下划线和连字符
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.
    
    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [2]:
## 自定义工具属性
### 自定义工具名称
@tool("web_search")     # 自定义名称
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)      # 输出 web_search

web_search


In [3]:
### 自定义工具描述(类似 skills 的触发规则)
@tool("calculator", description="Performs arithmetic calculators. Use this for math prol=blems.")
def calc(expression: str) -> str:
    """Evaluate mathmatical expressions."""
    return str(eval(expression))

**① 用户输入**  
你在 Claude Code 对话框发送：*"上海的天气怎么样？用摄氏度显示"*

**② 大模型读取工具说明书**  
Claude(模型) 分析意图后，查阅系统提示词中由 `@tool` 生成的 `WeatherInput` 类说明书（包含 `location` 必填、`units` 默认 `celsius` 等描述）。

**③ 大模型生成参数 JSON**  
Claude(模型) 根据说明书输出调用参数（省略了有默认值的字段）：  
`{"location": "上海"}`

**④ Agent 框架校验并实例化**  
后台捕获 JSON，用 `WeatherInput` 类验证，自动补全缺失的默认值：  
`WeatherInput(location="上海", units="celsius", include_forecast=False)`

**⑤ 框架调用你的函数**  
框架将 Pydantic 对象拆解，实际执行：  
`get_weather(location="上海", units="celsius", include_forecast=False)`

**⑥ 函数执行业务逻辑**  
- 判断 `units == "celsius"`，设 `temp = 22`  
- 拼接字符串（此时 `.upper()` 正确执行，取 `"celsius"[0].upper()` 得到 `"C"`）  
- `include_forecast` 为 `False`，不追加预报内容  
- 结果暂存为：`"Current weather in 上海: 22 degrees C"`

**⑦ 函数返回原始结果**  
`get_weather` 将上述字符串返回给 Agent 框架。

**⑧ 大模型翻译并回复用户**  
框架将字符串传给 Claude，Claude 重新组织成自然语言输出到你的聊天框：  
*"上海当前气温为 22 摄氏度。"*

In [4]:
## 高级模式自定义
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):          # 继承 创建数据模型的基类 BaseModel(让定义的 WeatherInput 类具备自动校验参数类型、必填项的能力)
    """Input for weather queries."""
    # location 为 str 类型的属性，Field 函数 为模型字段添加额外元数据，这里添加 description，让描述可以被大模型读取
    location: str = Field(
        description="City name or coordinates"  # 给大模型看的提示语，告诉模型“请在这里填城市名或坐标”
    )
    # Literal:精确限定值范围，这里明确限制 单位(units) 为摄氏度(celsius)或者华氏度(fahrenheit)
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", 
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecase"
    )

# 这个装饰器告诉底层的 AI 框架：“下面的 get_weather 函数是一个可被大模型调用的工具，传入参数的格式请严格按照 WeatherInput 这个类的定义来解析
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) ->str:   # 参数列表要与 WeatherInput 保持一致
    """Get current weather and optional forecase."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:    # 如果用户要求包含预报，就在结果末尾追加接下来五天的天气（这里硬编码为晴朗）
        result += "\nNext 5 days: Sunny"
    return result

## 保留参数名称 -- config 和 runtime 参数不能用于工具参数，否则会导致运行错误